In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
# Edit these paths before running

CSV_PATH   = r"data/raw/wc2026_canada_vs_bosnia_and_herzegovina_2026-06-12_events.csv"
MODEL_PATH = r"model/xg_sal313_lr.pkl"
FONT_DIR   = r"assets/fonts"
OUTPUT_DIR = r"."

COMPETITION = "FIFA World Cup 2026"
ROUND       = "Group Stage"


In [ ]:
import os, glob, pickle, joblib, platform, requests, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.font_manager as fm
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from matplotlib.image import imread
from mplsoccer import Pitch
from PIL import Image
from io import BytesIO

warnings.filterwarnings("ignore")
matplotlib.rcParams["figure.dpi"] = 150

# Fonts
if os.path.exists(FONT_DIR):
    for ttf in glob.glob(os.path.join(FONT_DIR, "**", "*.ttf"), recursive=True):
        fm.fontManager.addfont(ttf)
    fm._load_fontmanager(try_read_cache=False)
    BEBAS  = "Bebas Neue"
    DMSANS = "DM Sans"
else:
    print(f"⚠  Font dir not found: {FONT_DIR}. Download Bebas Neue + DM Sans from Google Fonts.")
    BEBAS  = "DejaVu Sans"
    DMSANS = "DejaVu Sans"

# Model
try:    _obj = joblib.load(MODEL_PATH)
except: _obj = pickle.load(open(MODEL_PATH, "rb"))
_model, _scaler = _obj["model"], _obj["scaler"]
GOAL_Y1, GOAL_Y2, GOAL_Y_CENTER, GOAL_X = 36.0, 44.0, 40.0, 120.0

# Colors
BG         = "#FFFFFF"
PITCH_LINE = "#CCCCCC"
TEXT_DARK  = "#1A1A1A"
TEXT_LIGHT = "#999999"
DIVIDER    = "#E8E8E8"

TEAM_META = {
    "Mexico": ("#CE1126", "mx"), "South Africa": ("#007A4D", "za"),
    "South Korea": ("#C60C30", "kr"), "Republic of Korea": ("#C60C30", "kr"),
    "Korea Republic": ("#C60C30", "kr"), "Czechia": ("#002395", "cz"),
    "Czech Republic": ("#002395", "cz"), "Canada": ("#FF0000", "ca"),
    "Bosnia and Herzegovina": ("#002395", "ba"), "Bosnia & Herzegovina": ("#002395", "ba"),
    "Qatar": ("#8D1B3D", "qa"), "Switzerland": ("#FF0000", "ch"),
    "Brazil": ("#009C3B", "br"), "Morocco": ("#C1272D", "ma"),
    "Haiti": ("#00209F", "ht"), "Scotland": ("#003399", "gb-sct"),
    "United States": ("#B22234", "us"), "USA": ("#B22234", "us"),
    "Paraguay": ("#D52B1E", "py"), "Australia": ("#00008B", "au"),
    "Türkiye": ("#E30A17", "tr"), "Turkey": ("#E30A17", "tr"),
    "Germany": ("#1a1a1a", "de"), "Curaçao": ("#002B7F", "cw"),
    "Curacao": ("#002B7F", "cw"), "Ivory Coast": ("#F77F00", "ci"),
    "Côte d'Ivoire": ("#F77F00", "ci"), "Ecuador": ("#FFD100", "ec"),
    "Netherlands": ("#FF6600", "nl"), "Japan": ("#BC002D", "jp"),
    "Sweden": ("#006AA7", "se"), "Tunisia": ("#E70013", "tn"),
    "Belgium": ("#000000", "be"), "Egypt": ("#C8102E", "eg"),
    "Iran": ("#239F40", "ir"), "IR Iran": ("#239F40", "ir"),
    "New Zealand": ("#00247D", "nz"), "Spain": ("#AA151B", "es"),
    "Cape Verde": ("#003893", "cv"), "Cabo Verde": ("#003893", "cv"),
    "Saudi Arabia": ("#006C35", "sa"), "Uruguay": ("#75AADB", "uy"),
    "France": ("#002395", "fr"), "Senegal": ("#00853F", "sn"),
    "Iraq": ("#CE1126", "iq"), "Norway": ("#EF2B2D", "no"),
    "Argentina": ("#75AADB", "ar"), "Algeria": ("#006233", "dz"),
    "Austria": ("#ED2939", "at"), "Jordan": ("#007A3D", "jo"),
    "Portugal": ("#006600", "pt"), "DR Congo": ("#007FFF", "cd"),
    "Congo DR": ("#007FFF", "cd"), "Uzbekistan": ("#1EB53A", "uz"),
    "Colombia": ("#FCD116", "co"), "England": ("#CF081F", "gb-eng"),
    "Croatia": ("#FF0000", "hr"), "Ghana": ("#EF3340", "gh"),
    "Panama": ("#DA121A", "pa"),
}


In [ ]:
def is_true(v):
    return v is True or str(v).strip().lower() == "true"

def build_xg_features(row):
    px = row["x"] * 1.2
    py = (100 - row["y"]) * 0.8
    dist      = np.sqrt((px - GOAL_X)**2 + (py - GOAL_Y_CENTER)**2)
    angle_rad = abs(np.arctan2(abs(py - GOAL_Y_CENTER), max(GOAL_X - px, 0.1)))
    denom     = px**2 + (py - GOAL_Y_CENTER)**2 - ((GOAL_Y2 - GOAL_Y1) / 2)**2
    vis_angle = max(np.arctan2((GOAL_Y2 - GOAL_Y1) * px, denom), 0)
    return [dist, angle_rad, vis_angle, abs(py - GOAL_Y_CENTER),
            float(is_true(row.get("qual_Head",          False))),
            float(is_true(row.get("qual_BigChance",     False))),
            float(is_true(row.get("qual_FastBreak",     False))),
            float(is_true(row.get("qual_SetPiece",      False)) or
                  is_true(row.get("qual_FreekickTaken", False))),
            float(is_true(row.get("qual_Assisted",      False))),
            0.0,
            float(is_true(row.get("qual_Penalty",       False))),
            float(is_true(row.get("qual_RightFoot",     False))),
            float(is_true(row.get("qual_Head",          False))) * dist,
            float(is_true(row.get("qual_BigChance",     False))) * dist]

def score_shots(shots_df):
    feats    = np.array([build_xg_features(r) for _, r in shots_df.iterrows()])
    X_scaled = _scaler.transform(feats)
    xg       = _model.predict_proba(X_scaled)[:, 1]
    for i, (_, r) in enumerate(shots_df.iterrows()):
        if is_true(r.get("qual_Penalty", False)):
            xg[i] = 0.76
    return xg

def get_flag(code, size=160):
    url = f"https://flagcdn.com/w{size}/{code}.png"
    try:
        r = requests.get(url, timeout=8, headers={"User-Agent": "Mozilla/5.0"})
        if r.status_code == 200:
            return Image.open(BytesIO(r.content)).convert("RGBA")
    except: pass
    return None

def add_flag(ax, img, xy, zoom=0.18, zorder=5):
    if img is None: return
    oi = OffsetImage(np.array(img), zoom=zoom)
    ab = AnnotationBbox(oi, xy, frameon=False, zorder=zorder, box_alignment=(0.5, 0.5))
    ax.add_artist(ab)

def xg_to_size(xg):
    return 18 + (xg - 0.01) / (0.99 - 0.01) * (200 - 18)

def format_goalscorers(gdf):
    if len(gdf) == 0: return []
    seen = {}
    for _, row in gdf.iterrows():
        name        = row["player"]
        display_min = int(row["minute"]) + 1
        if display_min > 90:   minute_str = f"90+{display_min - 90}'"
        elif display_min > 45 and row.get("period_name") == "FirstHalf":
                               minute_str = f"45+{display_min - 45}'"
        else:                  minute_str = f"{display_min}'"
        if name not in seen: seen[name] = []
        seen[name].append(minute_str)
    return [f"{name} {', '.join(minutes)}" for name, minutes in seen.items()]


In [ ]:
df         = pd.read_csv(CSV_PATH)
home_team  = df["home_team"].iloc[0]
away_team  = df["away_team"].iloc[0]
home_score = int(df["home_score"].iloc[0])
away_score = int(df["away_score"].iloc[0])

_dt        = pd.to_datetime(df["match_date"].iloc[0])
match_date = _dt.strftime("%d %B %Y").lstrip("0") if platform.system() == "Windows"              else _dt.strftime("%-d %B %Y")

home_color    = TEAM_META.get(home_team, ("#333333", "un"))[0]
away_color    = TEAM_META.get(away_team, ("#333333", "un"))[0]
home_flag_img = get_flag(TEAM_META.get(home_team, ("", "un"))[1])
away_flag_img = get_flag(TEAM_META.get(away_team, ("", "un"))[1])

shots       = df[df["isShot"] == True].copy().reset_index(drop=True)
shots["xg"] = score_shots(shots)
home_shots  = shots[shots["team"] == home_team].copy()
away_shots  = shots[shots["team"] == away_team].copy()

passes  = df[(df["event"] == "Pass") & (df["outcome"] == "Successful")]
corners = df[(df["event"] == "CornerAwarded") & (df["outcome"] == "Successful")]
on_tgt  = df[
    (df["isShot"] == True) &
    (df["event"].isin(["SavedShot", "Goal"])) &
    (df.get("qual_Blocked", pd.Series(False, index=df.index)) != True)
]

def ts(sub, team): return len(sub[sub["team"] == team])

home_xg = round(home_shots["xg"].sum(), 2)
away_xg = round(away_shots["xg"].sum(), 2)

STATS = [
    ("xG",        home_xg,               away_xg,               True),
    ("SHOTS",     ts(shots, home_team),   ts(shots, away_team),  False),
    ("ON TARGET", ts(on_tgt, home_team),  ts(on_tgt, away_team), False),
    ("CORNERS",   ts(corners, home_team), ts(corners, away_team),False),
    ("PASSES",    ts(passes, home_team),  ts(passes, away_team), False),
]

goals_df        = df[df["isGoal"] == True].copy().sort_values("minute")
home_goal_lines = format_goalscorers(goals_df[goals_df["team"] == home_team])
away_goal_lines = format_goalscorers(goals_df[goals_df["team"] == away_team])
max_scorers     = max(len(home_goal_lines), len(away_goal_lines))

print(f"{home_team} {home_score}–{away_score} {away_team}  ·  {match_date}")
print(f"Home xG: {home_xg}  |  Away xG: {away_xg}")


In [ ]:
L_PAD    = 0.14
R_PAD    = 0.14
CONTENT_W = 1.0 - L_PAD - R_PAD
HEADER_H  = 0.155
PITCH_B   = 0.02

if max_scorers == 0:  GAP = 0.012
elif max_scorers == 1: GAP = 0.026
elif max_scorers == 2: GAP = 0.052
else:                  GAP = 0.012 + max_scorers * 0.018

PITCH_H = 1.0 - HEADER_H - GAP - PITCH_B - 0.01

FIG_W, FIG_H = 1600/150, 900/150
fig = plt.figure(figsize=(FIG_W, FIG_H), facecolor=BG)

# Header
ax_hdr = fig.add_axes([L_PAD, 1 - HEADER_H, CONTENT_W, HEADER_H])
ax_hdr.set_facecolor(BG); ax_hdr.set_xlim(0,1); ax_hdr.set_ylim(0,1); ax_hdr.axis("off")
ax_hdr.axhline(0.04, color=DIVIDER, linewidth=1.2)
ax_hdr.text(0.5, 0.65, f"{home_score}  –  {away_score}",
            ha="center", va="center", fontfamily=BEBAS, fontsize=44, color=TEXT_DARK,
            transform=ax_hdr.transAxes)
name_fontsize = 24 if max(len(home_team), len(away_team)) <= 12 else                 18 if max(len(home_team), len(away_team)) <= 18 else 14
ax_hdr.text(0.40, 0.65, home_team.upper(), ha="right", va="center",
            fontfamily=BEBAS, fontsize=name_fontsize, color=home_color,
            transform=ax_hdr.transAxes)
ax_hdr.text(0.60, 0.65, away_team.upper(), ha="left", va="center",
            fontfamily=BEBAS, fontsize=name_fontsize, color=away_color,
            transform=ax_hdr.transAxes)
char_w = 0.018 if name_fontsize==24 else 0.014 if name_fontsize==18 else 0.011
add_flag(ax_hdr, home_flag_img, (0.40 - len(home_team)*char_w - 0.04, 0.65), zoom=0.22)
add_flag(ax_hdr, away_flag_img, (0.60 + len(away_team)*char_w + 0.01, 0.65), zoom=0.22)
ax_hdr.text(0.5, 0.2, f"{COMPETITION}  ·  {ROUND}  ·  {match_date}",
            ha="center", va="center", fontfamily=DMSANS, fontsize=8.5, color=TEXT_LIGHT,
            transform=ax_hdr.transAxes)

# Goalscorers
if max_scorers > 0:
    ax_gap = fig.add_axes([L_PAD, 1 - HEADER_H - GAP, CONTENT_W, GAP])
    ax_gap.set_xlim(0,1); ax_gap.set_ylim(0,1); ax_gap.axis("off"); ax_gap.set_facecolor(BG)
    LINE_STEP = 1.0 / (max_scorers + 0.25)
    ax_gap.text(0.5, 1.0 - LINE_STEP, "⚽", ha="center", va="center",
                fontsize=9, fontfamily="Segoe UI Emoji", transform=ax_gap.transAxes)
    for j, line in enumerate(home_goal_lines):
        ax_gap.text(0.47, 1.0 - (j+1)*LINE_STEP, line, ha="right", va="center",
                    fontfamily=DMSANS, fontsize=7.5, color=TEXT_DARK,
                    transform=ax_gap.transAxes)
    for j, line in enumerate(away_goal_lines):
        ax_gap.text(0.53, 1.0 - (j+1)*LINE_STEP, line, ha="left", va="center",
                    fontfamily=DMSANS, fontsize=7.5, color=TEXT_DARK,
                    transform=ax_gap.transAxes)

# Pitch
pitch = Pitch(pitch_type="opta", pitch_color=BG, line_color=PITCH_LINE,
              linewidth=1.2, goal_type="box", corner_arcs=True, line_zorder=2)
ax_pitch = fig.add_axes([L_PAD, PITCH_B, CONTENT_W, PITCH_H])
pitch.draw(ax=ax_pitch); ax_pitch.set_facecolor(BG)


ax_pitch.add_patch(mpatches.Rectangle((0,0),50,100,color=home_color,alpha=0.03,zorder=0))
ax_pitch.add_patch(mpatches.Rectangle((50,0),50,100,color=away_color,alpha=0.03,zorder=0))

def draw_shots(shot_df, color, side):
    for _, s in shot_df.iterrows():
        px = max(1.5, min(48.5, 100-s["x"])) if side=="home" else max(51.5, min(98.5, s["x"]))
        size = xg_to_size(s["xg"])
        if s["event"] == "Goal":
            ax_pitch.scatter(px, s["y"], s=size, c=color, alpha=1.0,
                             edgecolors=color, linewidths=0.5, marker="o", zorder=5)
        else:
            ax_pitch.scatter(px, s["y"], s=size, facecolors="none",
                             edgecolors=color, linewidths=1.4, marker="o", zorder=5, alpha=0.85)

draw_shots(home_shots, home_color, "home")
draw_shots(away_shots, away_color, "away")

# Stats overlay
N=len(STATS); CX=50.0; CY=50.5; BLOCK_H=56.0; ROW_H=BLOCK_H/N; TOP_Y=CY+BLOCK_H/2-ROW_H*0.5
BAR_LEFT=33.5; BAR_RIGHT=66.5; BAR_W=BAR_RIGHT-BAR_LEFT; NUM_PAD=1.8; BAR_H_PTS=ROW_H*0.40

for i, (label, h_val, a_val, is_float) in enumerate(STATS):
    row_cy=TOP_Y-i*ROW_H; bar_y=row_cy-ROW_H*0.06
    total=max(float(h_val)+float(a_val),0.001); home_frac=float(h_val)/total
    home_w=home_frac*BAR_W; away_w=BAR_W-home_w
    ax_pitch.add_patch(mpatches.Rectangle((BAR_LEFT,bar_y-BAR_H_PTS/2),BAR_W,BAR_H_PTS,
                                           facecolor="#E0E0E0",edgecolor="none",zorder=6))
    if home_w>0.1:
        ax_pitch.add_patch(mpatches.Rectangle((BAR_LEFT,bar_y-BAR_H_PTS/2),home_w,BAR_H_PTS,
                                               facecolor=home_color,edgecolor="none",alpha=0.88,zorder=7))
    if away_w>0.1:
        ax_pitch.add_patch(mpatches.Rectangle((BAR_LEFT+home_w,bar_y-BAR_H_PTS/2),away_w,BAR_H_PTS,
                                               facecolor=away_color,edgecolor="none",alpha=0.88,zorder=7))
    ax_pitch.text(CX,bar_y,label,ha="center",va="center",fontfamily=DMSANS,fontsize=6.8,
                  fontweight="bold",color="white",zorder=9)
    fmt_h=f"{h_val:.2f}" if is_float else str(int(h_val))
    fmt_a=f"{a_val:.2f}" if is_float else str(int(a_val))
    ax_pitch.text(BAR_LEFT-NUM_PAD,bar_y,fmt_h,ha="right",va="center",
                  fontfamily=BEBAS,fontsize=17,color=home_color,zorder=8)
    ax_pitch.text(BAR_RIGHT+NUM_PAD,bar_y,fmt_a,ha="left",va="center",
                  fontfamily=BEBAS,fontsize=17,color=away_color,zorder=8)
    if i<N-1:
        ax_pitch.plot([BAR_LEFT-1,BAR_RIGHT+1],[row_cy-ROW_H*0.5]*2,
                      color="#EEEEEE",linewidth=0.5,zorder=6)

# Legend
ax_pitch.scatter(43,8,s=35,facecolors="none",edgecolors=TEXT_LIGHT,linewidths=1.2,marker="o",zorder=5)
ax_pitch.text(44.2,8,"Shot",ha="left",va="center",fontfamily=DMSANS,fontsize=6.5,color=TEXT_LIGHT,zorder=5)
ax_pitch.scatter(43,13.5,s=35,c=TEXT_LIGHT,alpha=1.0,edgecolors=TEXT_LIGHT,linewidths=0.5,marker="o",zorder=5)
ax_pitch.text(44.2,13.5,"Goal",ha="left",va="center",fontfamily=DMSANS,fontsize=6.5,color=TEXT_LIGHT,zorder=5)

xg_sizes=[xg_to_size(v) for v in [0.05,0.20,0.50,0.99]]
for k,sz in enumerate(xg_sizes):
    ax_pitch.scatter(53+k*4.5,10.5,s=sz,facecolors="none",edgecolors=TEXT_LIGHT,linewidths=1.0,marker="o",zorder=5)
ax_pitch.text(52.7,5.5,"Low xG",ha="left",va="center",fontfamily=DMSANS,fontsize=6.5,color=TEXT_LIGHT,zorder=5)
ax_pitch.text(53+3*4.5+0.3,5.5,"High xG",ha="right",va="center",fontfamily=DMSANS,fontsize=6.5,color=TEXT_LIGHT,zorder=5)

              fontweight="bold",color=TEXT_DARK,zorder=5,clip_on=False)
              fontweight="bold",color=TEXT_DARK,zorder=5,clip_on=False)
ax_pitch.text(98,1.5,"Data: WhoScored",ha="right",va="bottom",fontfamily=DMSANS,fontsize=6,
              color=TEXT_LIGHT,zorder=5)

# Save
safe_home = home_team.lower().replace(" ","_")
safe_away = away_team.lower().replace(" ","_")
date_str  = df["match_date"].iloc[0]
out_path  = os.path.join(OUTPUT_DIR, f"wc_postmatch_{safe_home}_vs_{safe_away}_{date_str}.png")
plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor=BG, edgecolor="none")
print(f"✓ Saved → {out_path}")
plt.show()
